# Chapter 8 — Task Decomposition Strategies · Homework (self-test)

**Goal:** check that you can pick the right decomposition for a task, build multi-pass and independent review requests, write a bounded async fan-out that survives failures, calibrate confidence routing, and control an adaptive planner loop.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 90–120 minutes
- Estimated API cost: **$0** (everything graded runs offline against a fake API); the one optional live cell costs about $0.01
- Prerequisites: the practice notebook, and the venv from [00-prerequisites](../../00-prerequisites/README.md)

**How to use it**
1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. Part A: answer 10 exam-style questions in the `answers` cell, then run the grader.
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. Part C: write a short design answer, then compare it with the rubric.
5. Run the **Scorecard**. Pass mark: **72%** overall (quiz + exercises), a rough stand-in for the exam's scaled 720/1000 (the real raw-to-scaled conversion is not published).

In [ ]:
import asyncio
import hashlib
import json
import os
import re
import time
from itertools import count
from typing import Any, Callable, Literal

import anthropic
import httpx2
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
HAS_KEY = bool(os.getenv("ANTHROPIC_API_KEY"))   # only the optional live cell needs it
aclient = anthropic.AsyncAnthropic()   # only the optional live exercise uses it


def parsed_or_raise(response):
    """Structured outputs guarantee shape only when the model finished normally."""
    if response.stop_reason != "end_turn" or response.parsed_output is None:
        raise RuntimeError(f"step failed: stop_reason={response.stop_reason}")
    return response.parsed_output


print("anthropic", anthropic.__version__, "| MODEL =", MODEL, "| API key found:", HAS_KEY)

The optional live cell needs a key. Without one, the next cell only prints a notice: everything graded runs offline.

In [ ]:
if HAS_KEY:
    print("ANTHROPIC_API_KEY found: the optional live cell will run.")
else:
    print("No ANTHROPIC_API_KEY: the optional live cell will be skipped. Everything graded runs offline.")

### Helpers (run, don't edit)

`FakeClaude` is the same offline API as in the practice notebook: real `Anthropic` / `AsyncAnthropic` clients whose HTTP layer replays scripted replies, records every request in `.requests` and the peak concurrency in `.max_in_flight`. The schemas and prompts below are the README's.

In [ ]:
# ---------- grading bookkeeping ----------
RESULTS: dict[int, bool] = {}
SCORE: dict[str, int] = {}


def _record(n: int, exc: BaseException | None) -> None:
    if exc is None:
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    elif isinstance(exc, NotImplementedError):
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    elif isinstance(exc, AssertionError):
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    else:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {str(exc)[:200]}")


def _run_check(n: int, check: Callable[[], None]) -> None:
    """Run one checker; never let it crash the notebook."""
    try:
        check()
        _record(n, None)
    except Exception as exc:  # noqa: BLE001 - report, don't crash
        _record(n, exc)


async def _arun_check(n: int, check) -> None:
    """Async version for checkers that await your code."""
    try:
        await check()
        _record(n, None)
    except Exception as exc:  # noqa: BLE001
        _record(n, exc)


# ---------- offline Messages API (real SDK clients, fake HTTP layer) ----------

_ids = count(1)


def reply(content: str | list[dict], stop_reason: str = "end_turn") -> dict:
    """A Messages API response body (wire format). `content` is a text string or a list of blocks."""
    blocks = [{"type": "text", "text": content}] if isinstance(content, str) else content
    return {"id": f"msg_{next(_ids):04d}", "type": "message", "role": "assistant", "model": "scripted",
            "content": blocks, "stop_reason": stop_reason, "stop_sequence": None}


def reply_json(obj: Any, stop_reason: str = "end_turn") -> dict:
    """A reply whose text is JSON, as structured outputs (`messages.parse`) return it."""
    data = obj.model_dump() if isinstance(obj, BaseModel) else obj
    return reply(json.dumps(data), stop_reason=stop_reason)


def api_error(status: int, err_type: str, message: str, headers: dict | None = None) -> dict:
    """A scripted HTTP error, e.g. api_error(429, "rate_limit_error", "...", {"retry-after-ms": "50"})."""
    return {"_status": status, "_headers": headers or {}, "type": "error",
            "error": {"type": err_type, "message": message}}


def _texts(content: Any) -> list[str]:
    if isinstance(content, str):
        return [content]
    return [b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text"]


def system_text(body: dict) -> str:
    return "".join(_texts(body.get("system", "")))


def user_text(body: dict) -> str:
    """Text of the LAST user message in a request body."""
    users = [m for m in body["messages"] if m["role"] == "user"]
    return "".join(_texts(users[-1]["content"])) if users else ""


def _tokens(s: str) -> int:
    return len(s) // 4 + 1          # rough estimate; real counts come from the API's `usage`


class FakeClaude:
    """Offline Messages API with real SDK clients on top.

    script: a list of replies served in order, or a function `body -> reply` (use a function for
            concurrent calls, whose arrival order is not deterministic).
    .client / .aclient: real `anthropic.Anthropic` / `AsyncAnthropic` whose HTTP transport is this fake,
            so SDK retries, `messages.parse()` and response types all run for real.
    Records every request body in .requests, every usage in .usages, and the peak number of concurrent requests in .max_in_flight.
    Simulates prompt caching: a prefix ending in `cache_control` is written on first use and can be
    read only by requests that ARRIVE after an earlier response with that prefix has begun.
    """

    def __init__(self, script, latency: float = 0.0, max_retries: int = 2, min_cacheable: int = 512):
        self.script = script if callable(script) else list(script)
        self.latency, self.min_cacheable = latency, min_cacheable
        self.requests: list[dict] = []
        self.usages: list[dict] = []          # usage of every successful response, in completion order
        self.in_flight = self.max_in_flight = 0
        self._cache_ready: set[str] = set()
        self.client = anthropic.Anthropic(
            api_key="offline", max_retries=max_retries,
            http_client=anthropic.DefaultHttpxClient(transport=httpx2.MockTransport(self._handle_sync)))
        self.aclient = anthropic.AsyncAnthropic(
            api_key="offline", max_retries=max_retries,
            http_client=anthropic.DefaultAsyncHttpxClient(transport=httpx2.MockTransport(self._handle_async)))

    def _next(self, body: dict) -> dict:
        if callable(self.script):
            return dict(self.script(body))
        if not self.script:
            return api_error(500, "api_error", "FakeClaude: script exhausted (did a loop not stop?)")
        return dict(self.script.pop(0))

    def _prefix(self, body: dict) -> str | None:
        """Text of the system prompt up to the last block marked with cache_control (if any)."""
        system = body.get("system")
        if not isinstance(system, list):
            return None
        marked = [i for i, b in enumerate(system) if b.get("cache_control")]
        return "".join(b.get("text", "") for b in system[: marked[-1] + 1]) if marked else None

    def _usage(self, body: dict, out: dict, cache_hit: bool) -> dict:
        total_in = _tokens(json.dumps(body.get("system", "")) + json.dumps(body["messages"]))
        prefix = self._prefix(body)
        cached = _tokens(prefix) if prefix and _tokens(prefix) >= self.min_cacheable else 0
        usage = {"input_tokens": total_in - cached, "output_tokens": _tokens("".join(_texts(out["content"]))),
                 "cache_creation_input_tokens": 0 if cache_hit else cached,
                 "cache_read_input_tokens": cached if cache_hit else 0}
        if cached:
            self._cache_ready.add(prefix)     # the response has begun: the entry is now readable
        self.usages.append(usage)
        return usage

    def _response(self, body: dict, out: dict, cache_hit: bool) -> httpx2.Response:
        status, headers = out.pop("_status", 200), out.pop("_headers", {})
        out.pop("_delay", None)
        if status == 200:
            out.setdefault("usage", self._usage(body, out, cache_hit))
        return httpx2.Response(status, json=out, headers=headers)

    def _handle_sync(self, request: httpx2.Request) -> httpx2.Response:
        body = json.loads(request.content)
        self.requests.append(body)
        cache_hit = self._prefix(body) in self._cache_ready
        return self._response(body, self._next(body), cache_hit)

    async def _handle_async(self, request: httpx2.Request) -> httpx2.Response:
        body = json.loads(request.content)
        self.requests.append(body)
        cache_hit = self._prefix(body) in self._cache_ready      # decided when the request ARRIVES
        self.in_flight += 1
        self.max_in_flight = max(self.max_in_flight, self.in_flight)
        try:
            out = self._next(body)
            await asyncio.sleep(out.get("_delay", self.latency))
        finally:
            self.in_flight -= 1
        return self._response(body, out, cache_hit)


# ---------- schemas and prompts used by the exercises (same as the README) ----------
class Finding(BaseModel):
    file: str
    line: int | None
    severity: Literal["critical", "high", "medium", "low"]
    issue: str
    suggested_fix: str
    evidence: str


class FileReview(BaseModel):
    findings: list[Finding]
    exports: list[str]
    imports: list[str]


class CrossFileReview(BaseModel):
    findings: list[Finding]


class Review(BaseModel):
    approved: bool
    findings: list[Finding]


class WorkerBrief(BaseModel):
    id: str
    objective: str
    boundaries: str
    output_format: str


class WorkerResult(BaseModel):
    task_id: str
    status: Literal["ok", "partial", "failed"]
    summary: str
    evidence: list[str]


class Subtask(BaseModel):
    goal: str
    rationale: str
    priority: Literal["high", "medium", "low"]


class Plan(BaseModel):
    done: bool
    next_subtasks: list[Subtask]
    open_questions: list[str]


PATTERNS = ("augmented_llm", "routing", "prompt_chaining", "sectioning", "evaluator_optimizer", "agent",
            "orchestrator_workers")
LOCAL_RULES = ("Review ONE file. Report only issues visible inside this file. Use the severity rubric exactly. "
               "Quote the code each finding is about in `evidence`.")
INTEGRATION_RULES = ("You receive each file's public interface and its local findings. Report ONLY issues that cross "
                     "file boundaries: contracts, types, data flow, error propagation, circular dependencies.")
REVIEWER_SYSTEM = ("You are a code reviewer who did not write this change. Check it against the spec and the criteria. "
                   "Assume nothing is correct until you have checked it. Report only issues you can point to in the diff.")
WORKER_SYSTEM = "You are a focused worker. Do only your brief. Stay inside its boundaries. Return status 'partial' if you cannot finish."

print("helpers ready")

## Part A — Quiz (10 questions)

Pick one letter per question. Scenarios follow the exam's style: Claude Code for CI, developer productivity, multi-agent research, customer support, structured extraction and solution design.

**Q1. (Developer productivity)** You ask Claude Code to migrate an unfamiliar 200-module monorepo from one logging library to another. Halfway through, it discovers that 40 modules log through an in-house wrapper nobody mentioned, and those modules need a different change. Which decomposition approach fits this task?

- A) Write a fixed four-step chain up front (find usages, replace calls, run tests, commit) and restart it from scratch whenever something unexpected appears.
- B) Map how each module logs before changing anything, group modules by logging style, plan per group, and add a new subtask for the wrapper when it is discovered.
- C) Send the whole monorepo to one call on the model with the largest context window so it sees everything at once.
- D) Run three independent full-repo migrations and keep only the changes that at least two of them agree on.

**Q2. (Structured data extraction)** Your invoice pipeline runs extract → enrich (vendor lookup) → post to the ERP. Occasionally an extraction whose line items do not add up to the stated total is posted anyway. Which change fixes this in the right place?

- A) Add "double-check that the totals add up" to the enrichment prompt.
- B) Replace the pipeline with an autonomous agent that decides when validation is needed.
- C) Raise `max_tokens` on the extraction step so the model has more room to reason.
- D) Add a gate in code after extraction that sums the line items, compares them with the stated total and routes mismatches to human review before enrichment.

**Q3. (Claude Code for CI)** You already run per-file local review passes on a 14-file PR and are now designing the integration pass. What should it receive?

- A) Each file's public interface (signatures, exports and imports), the local findings and the PR description, plus a tool to read a specific function when it needs one.
- B) The full source of all 14 files again plus the local findings, so it has complete context.
- C) Only the PR description, because the local passes have already seen the code.
- D) The full text of the local reviews and the diff of the largest file, since most bugs are there.

**Q4. (Claude Code for CI)** A pipeline has Claude generate a database migration, then appends "Now review this migration critically" to the same conversation. The review approves almost everything; humans later find problems that the generator had explicitly considered and dismissed. What is the most effective change?

- A) Raise the effort level for the review turn so it thinks longer.
- B) Make the review instruction stricter: "Be harsh and assume there are bugs."
- C) Send the migration, the ticket and the review criteria to a new request with a reviewer system prompt and none of the generation conversation.
- D) Have the generator write unit tests for its migration in the same session, then review again.

**Q5. (Claude Code for CI)** Your verification pass labels each finding high, medium or low confidence. On 400 labeled past findings, precision was 97% for high (n = 180), 74% for medium (n = 150) and 28% for low (n = 70). Policy requires at least 95% precision for inline PR comments. Which routing is best?

- A) Post every confirmed finding inline, sorted by confidence.
- B) Post confirmed high-confidence findings inline, send the other non-refuted findings to a triage queue, and keep spot-checking a sample of inline findings to detect drift.
- C) Post high and medium inline and silently drop low.
- D) Post high inline and silently drop everything else to keep the noise down.

**Q6. (Multi-agent research system)** Forty parallel worker calls on Claude Sonnet 5.5 share an identical 3,000-token system prompt marked with `cache_control`. They are all sent at the same moment with `asyncio.gather`. The usage logs show every call paid for a cache write and none read the cache. What is the most likely fix?

- A) Move `cache_control` onto each worker's unique brief instead of the shared prefix.
- B) Switch the workers to Claude Haiku 4.5, which has a lower minimum cacheable length.
- C) Disable SDK retries, because retries invalidate the cache entry.
- D) Send one request first and wait for its response to begin, then send the other 39 with the identical prefix.

**Q7. (Solution design)** A law firm receives about 2,000 intake emails a day. Each belongs to one of five practice areas, each area has a fixed intake form and process, and about 15% of emails are spam or unclear. A vendor proposes a coordinator agent with five specialist subagents. What do you recommend?

- A) A routing workflow: a fast model returns a structured category (with an other/unclear route to a human), and code dispatches each email to the existing process.
- B) The coordinator with five subagents, because it can adapt to unusual emails.
- C) One agent loop with tools for all five processes.
- D) Three independent classifiers per email with majority voting, followed by the coordinator agent.

**Q8. (Claude Code for CI)** Three differently prompted reviewers screen each code change for injection vulnerabilities. A missed vulnerability costs far more than a false alarm, and every flagged change goes through a verification pass afterwards. How should you combine their votes?

- A) Flag a change only when all three reviewers agree.
- B) Flag a change when at least two of the three agree.
- C) Flag a change when any reviewer flags it.
- D) Use only the reviewer with the best historical precision.

**Q9. (Customer support agent)** Your returns agent handles one-order messages well. When a customer writes about two orders at once, such as "A-1001 arrived broken and A-1002 never came", the logs show the right actions attached to the wrong orders: a carrier trace opened on A-1001, or a refund issued for A-1002. Which change best fixes this?

- A) Raise the agent's effort so it thinks longer before each tool call.
- B) Add a validator that rejects any tool call whose order ID does not appear in the customer's message, and retry the turn.
- C) Move the agent to the largest available model.
- D) Add few-shot examples that show multi-issue requests handled step by step, with each issue's order ID and parameters kept separate.

**Q10. (Solution design)** A team proposes splitting a refactor of a tightly coupled payments core across five parallel subagents. Every change touches the same shared module, and each subagent would need the others' changes to make progress. What is the strongest objection?

- A) Multi-agent setups fit poorly when every agent needs the same context and there are many dependencies between agents; a single agent or a sequential chain is simpler, cheaper and less error-prone here.
- B) Subagents cannot use tools, so they cannot edit files.
- C) Five subagents exceed the Messages API's limit on concurrent requests.
- D) Subagents share one context window, so five of them would overflow it.

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
    "q9": None,
    "q10": None,
}

In [ ]:
ANSWER_HASHES = {'q1': '96e2e2c94d02b534d4bdc303373df7b704ed4725981fac17549ee985d132c365', 'q2': 'd14b6ae0cbc51f4b715924684653e0d81035a8fe8b79de34ff29eb9db56e2d6b', 'q3': '2501e2fe43d858f1773d414ee14ab0a8cdf2afefbc4ffd20ae5d956e618f3ea3', 'q4': '69a9828043db3a177204403c987facd12c855928421b3aeeb5edce8407e3196c', 'q5': 'fbb43cbbf51767ad8ca178fa6cc4731661f1f1771789b9b7df93fba7ad386e2c', 'q6': '70425e0bfb4a573a030df7909d2bae441e3fdb1e7a565e097f3118231f4f27a9', 'q7': '8722338db4eb12eaae5f6f26c68875af2b115581d770f01f98b9e0e1e6010ac1', 'q8': '89443923f22dc808ba069cfb289117719cbe7d8d36703c74314c6b70fba0371b', 'q9': 'e58517837d895fe6365a5801be9a66d9d1d39f1d863920a84cbed549cfec1954', 'q10': 'aeb5147fa5276b8759290ab8af9ed3aaa01d2c844cecc89ea5547b013e209ad3'}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Coding exercises (offline)

Each exercise: read the task, replace the stub, run the checker. Exercise 4's checker is async (`await check_ex4()`), which Jupyter runs directly.

### Exercise 1 — Pick the pattern for a scenario

**(a)** Write `choose_pattern(profile)`. `profile` is a dict of booleans; a missing key means `False`. Apply the rules **in this order** and return the first match:

| # | Condition | Return |
|---|---|---|
| 1 | `one_call_enough` | `"augmented_llm"` |
| 2 | `distinct_categories` | `"routing"` |
| 3 | `steps_known` and `steps_depend` | `"prompt_chaining"` |
| 4 | `steps_known` (independent steps) | `"sectioning"` |
| 5 | `iterate_with_criteria` | `"evaluator_optimizer"` |
| 6 | `open_ended` (long, exploratory, environment-driven) | `"agent"` |
| 7 | otherwise (subtasks decided per request) | `"orchestrator_workers"` |

**(b)** Fill `SCENARIO_PROFILES` with a profile for each scenario so that `choose_pattern` returns the **best-fitting** pattern. The checker stores the expected patterns as hashes.

- **s1**: Every incoming vendor contract goes through the same five clause checks; each check uses the clause extracted by the previous step, and a final step assembles the report.
- **s2**: Support emails must go to billing, technical, refunds or a human queue; each destination already has its own process.
- **s3**: Marketing copy is translated into Japanese; a reviewer with a clear style-and-terminology rubric sends it back until it passes.
- **s4**: Investigate a production incident in a system nobody on the team knows: the number of steps is unknown and each step depends on logs and command output.
- **s5**: Twelve changed files are each reviewed with the same rubric; the files can be checked independently and the results merged.
- **s6**: Produce a market brief whose sub-questions depend on the client's question; a planner picks 3-6 research subtasks per request.
- **s7**: Turn one support ticket into a three-field summary.

Theory: [8.6](README.md#86-the-workflow-pattern-catalog), [8.7](README.md#87-workflow-or-agent-choosing-the-level-of-autonomy)

In [ ]:
def choose_pattern(profile: dict) -> str:
    """Return one of PATTERNS by applying the rule table in order."""
    # TODO
    raise NotImplementedError


SCENARIO_PROFILES = {   # TODO: one profile dict per scenario
    "s1": {},
    "s2": {},
    "s3": {},
    "s4": {},
    "s5": {},
    "s6": {},
    "s7": {},
}

In [ ]:
_SCENARIO_HASHES = {'s1': '9cb88d6cf746ed6e0a80e686607bbff09cb861a7945038c286bb58e7158f5da7', 's2': 'e0cafe4ef2120fe5895adea978e265372c01e3cc555011bac077d248c8b91a96', 's3': '8c63c8ad43e7296376787f6617604628f3fad2f22ae75f86caa5aea6c6716af6', 's4': '567021f57ecdeaaaee146cfd0d0dfb1b2ae946ded395c710a46f96af9be623d0', 's5': '45f0a0ac6d1343caa0adb37c904b7656609283dc9221882d6249fabc24d9796a', 's6': 'c696c998a69121a8d68c55db0ed6c60ac8346f8492cf47faa1ace45615f6e9fd', 's7': '988aab597bb88db3eae3ade90047fcd4f25d2cf996738a1416a750e9f755c7f3'}


def check_ex1() -> None:
    def check():
        cases = [
            ({"one_call_enough": True, "distinct_categories": True}, "augmented_llm"),
            ({"distinct_categories": True, "steps_known": True, "steps_depend": True}, "routing"),
            ({"steps_known": True, "steps_depend": True}, "prompt_chaining"),
            ({"steps_known": True}, "sectioning"),
            ({"steps_known": True, "iterate_with_criteria": True}, "sectioning"),
            ({"iterate_with_criteria": True, "open_ended": True}, "evaluator_optimizer"),
            ({"open_ended": True}, "agent"),
            ({}, "orchestrator_workers"),
        ]
        for profile, expected in cases:
            got = choose_pattern(profile)
            assert got == expected, f"(a) choose_pattern({profile}) returned {got!r}, expected {expected!r} (check the rule order)"
        for sid, digest in _SCENARIO_HASHES.items():
            profile = SCENARIO_PROFILES.get(sid)
            assert isinstance(profile, dict), f"(b) SCENARIO_PROFILES[{sid!r}] must be a dict"
            got = choose_pattern(profile)
            assert hashlib.sha256(f"{sid}:{got}".encode()).hexdigest() == digest, (
                f"(b) {sid}: your profile leads to {got!r}, which is not the best fit. Re-read the scenario and README 8.6-8.7.")
    _run_check(1, check)


check_ex1()

### Exercise 2 — Build the per-file and integration review requests

Write two functions that return **keyword-argument dicts** for `aclient.messages.parse(**kwargs)` (nothing is sent; the checker inspects the dicts):

`build_local_requests(files)`: `files` maps path → source. Return one dict per file, **in the same order**, each with
`model=MODEL`, a positive `max_tokens`, `system=LOCAL_RULES` (the same rubric for every file), `output_format=FileReview`, and
`messages=[{"role": "user", "content": f'<file path="{path}">\n{source}\n</file>'}]` (exactly one file per request).

`build_integration_request(local_reviews, pr_description)`: `local_reviews` maps path → `FileReview`. Return one dict with
`model=MODEL`, a positive `max_tokens`, `system=INTEGRATION_RULES`, `output_format=CrossFileReview` and one user message that contains the PR description and, **for every file**, its path, every export, every import and every local finding's `issue`. It must **not** contain the files' source code (that would recreate attention dilution).

Theory: [8.3](README.md#83-multi-pass-code-review)

In [ ]:
def build_local_requests(files: dict[str, str]) -> list[dict]:
    """One messages.parse kwargs dict per file (rules in the task above)."""
    # TODO
    raise NotImplementedError


def build_integration_request(local_reviews: dict[str, FileReview], pr_description: str) -> dict:
    """One messages.parse kwargs dict for the cross-file pass (interfaces + local findings, no source)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    def check():
        files = {
            "billing/invoice.py": "def total(lines):\n    # BODY_SENTINEL_1\n    return sum(l.amount for l in lines)\n",
            "billing/tax.py": "def vat(amount, rate):\n    # BODY_SENTINEL_2\n    return amount * rate\n",
            "api/handlers.py": "from billing.invoice import total\n# BODY_SENTINEL_3\ndef get_total(req):\n    return total(req.lines)\n",
        }
        reqs = build_local_requests(files)
        assert isinstance(reqs, list), f"return a list of request dicts, got {type(reqs).__name__}"
        assert len(reqs) == len(files), f"expected {len(files)} requests (one per file), got {len(reqs)}"
        for (path, source), req in zip(files.items(), reqs):
            assert req.get("system") == LOCAL_RULES, "every local request uses the same system=LOCAL_RULES (one shared rubric)"
            assert req.get("model") == MODEL and 0 < req.get("max_tokens", 0), "set model=MODEL and a positive max_tokens"
            assert req.get("output_format") is FileReview, "local passes return FileReview (exports/imports feed pass 2)"
            msgs = req.get("messages", [])
            assert len(msgs) == 1 and msgs[0].get("role") == "user", "exactly one user message per local request"
            content = msgs[0]["content"]
            assert f'<file path="{path}">' in content and source in content, f"request for {path} must contain its path tag and full source"
            others = [p for p in files if p != path and p in content]
            assert not others, f"the request for {path} also contains {others}: one file per local pass"

        local = {
            "billing/invoice.py": FileReview(exports=["total(lines) -> Decimal"], imports=[], findings=[
                Finding(file="billing/invoice.py", line=3, severity="medium", issue="float sum for money",
                        suggested_fix="use Decimal", evidence="sum(l.amount ...)")]),
            "billing/tax.py": FileReview(exports=["vat(amount, rate) -> float"], imports=[], findings=[]),
            "api/handlers.py": FileReview(exports=["get_total(req)"], imports=["billing.invoice.total"], findings=[
                Finding(file="api/handlers.py", line=4, severity="high", issue="req.lines may be missing",
                        suggested_fix="validate input", evidence="req.lines")]),
        }
        pr = "PR #812: switch invoice totals to Decimal"
        req = build_integration_request(local, pr)
        assert isinstance(req, dict), "return a dict of messages.parse kwargs"
        assert req.get("system") == INTEGRATION_RULES, "system must be INTEGRATION_RULES (cross-file issues only)"
        assert req.get("output_format") is CrossFileReview, "output_format must be CrossFileReview"
        assert req.get("model") == MODEL and 0 < req.get("max_tokens", 0), "set model=MODEL and a positive max_tokens"
        msgs = req.get("messages", [])
        assert len(msgs) == 1 and msgs[0].get("role") == "user", "one user message"
        content = msgs[0]["content"] if isinstance(msgs[0]["content"], str) else json.dumps(msgs[0]["content"])
        assert pr in content, "include the PR description"
        for path, review in local.items():
            assert path in content, f"missing file path {path}"
            for item in review.exports + review.imports + [f.issue for f in review.findings]:
                assert item in content, f"missing {item!r} for {path}: the integration pass needs every interface and local finding"
        leaked = [s for s in ("BODY_SENTINEL_1", "BODY_SENTINEL_2", "BODY_SENTINEL_3") if s in content]
        assert not leaked, "the integration request contains source code: send interfaces and findings, not the files again"
    _run_check(2, check)


check_ex2()

### Exercise 3 — Build an independent reviewer request

Write `build_independent_review(spec, diff, criteria, generator_messages)` returning `messages.parse` kwargs for a reviewer that is **independent** of the generator:

- `model=MODEL`, a positive `max_tokens`, `system=REVIEWER_SYSTEM`, `output_format=Review`;
- exactly **one** user message containing the spec, the criteria and the diff (wrap them in `<spec>`, `<criteria>`, `<diff>` tags);
- **nothing** from `generator_messages`: no assistant text, no thinking blocks, no author rationale. The argument is there to tempt you.

Theory: [8.4](README.md#84-independent-review-instances-why-self-review-is-weak)

In [ ]:
def build_independent_review(spec: str, diff: str, criteria: str, generator_messages: list[dict]) -> dict:
    """messages.parse kwargs for a reviewer that never sees the generator's conversation."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    def check():
        spec = "Add retry with exponential backoff to fetch_rates(); give up after 5 attempts."
        diff = "+for attempt in range(5):\n+    try:\n+        return fetch()\n+    except Timeout:\n+        time.sleep(2 ** attempt)"
        criteria = "Correct attempt count; no sleep after the last attempt; only retry transient errors."
        rationale = "AUTHOR_RATIONALE_7731: I checked the off-by-one case and the loop is correct."
        generator_messages = [
            {"role": "user", "content": spec},
            {"role": "assistant", "content": [
                {"type": "thinking", "thinking": "THINKING_SENTINEL_4410 sleeping after the last try is harmless", "signature": "sig"},
                {"type": "text", "text": rationale + "\n" + diff}]},
        ]
        req = build_independent_review(spec, diff, criteria, generator_messages)
        assert isinstance(req, dict), "return a dict of messages.parse kwargs"
        assert req.get("system") == REVIEWER_SYSTEM, "use system=REVIEWER_SYSTEM (a reviewer that did not write the code)"
        assert req.get("output_format") is Review, "output_format must be Review"
        assert req.get("model") == MODEL and 0 < req.get("max_tokens", 0), "set model=MODEL and a positive max_tokens"
        msgs = req.get("messages", [])
        dumped = json.dumps(msgs, default=str)
        assert "AUTHOR_RATIONALE_7731" not in dumped and "THINKING_SENTINEL_4410" not in dumped, (
            "the reviewer request contains the generator's reasoning: start a fresh request, do not reuse generator_messages")
        assert len(msgs) == 1 and msgs[0].get("role") == "user", (
            f"expected exactly one user message, got roles {[m.get('role') for m in msgs]}")
        content = msgs[0]["content"] if isinstance(msgs[0]["content"], str) else dumped
        for label, part in (("spec", spec), ("criteria", criteria), ("diff", diff)):
            assert part in content, f"the {label} is missing from the reviewer's message"
    _run_check(3, check)


check_ex3()

### Exercise 4 — A bounded async fan-out with partial failures

Write `async def fan_out(llm, briefs, concurrency)` that runs one worker call per `WorkerBrief` and returns a `list[WorkerResult]` **in the same order as `briefs`**:

- each call: `await llm.messages.parse(model=MODEL, max_tokens=1024, system=WORKER_SYSTEM, messages=[{"role": "user", "content": brief.model_dump_json()}], output_format=WorkerResult)`;
- run the calls **concurrently**, but never more than `concurrency` at a time (`asyncio.Semaphore`);
- trust the parsed result only when `stop_reason == "end_turn"` (use `parsed_or_raise`);
- never raise: an `anthropic.APIError`, a `ValueError` (pydantic validation error on truncated JSON) or a `RuntimeError` becomes `WorkerResult(task_id=brief.id, status="failed", summary=<error text>, evidence=[])`.

The checker drives your code with `FakeClaude` (latency 30 ms, one brief gets a 400, one is truncated by `max_tokens`) and measures the peak number of concurrent requests. It is async, so its cell uses `await`.

Theory: [8.8](README.md#88-fan-out-in-code-orchestrator-workers-with-asyncio-and-the-messages-api)

In [ ]:
async def fan_out(llm, briefs: list[WorkerBrief], concurrency: int) -> list[WorkerResult]:
    """Bounded concurrent worker calls; failures become WorkerResult(status="failed")."""
    # TODO
    raise NotImplementedError

In [ ]:
async def check_ex4() -> None:
    async def check():
        briefs = [WorkerBrief(id=f"b{i}", objective=f"objective {i}", boundaries="only this", output_format="bullets")
                  for i in range(1, 9)]

        def script(body: dict) -> dict:
            bid = json.loads(user_text(body))["id"]
            if bid == "b3":
                return api_error(400, "invalid_request_error", "brief too large")
            if bid == "b6":
                return reply('{"task_id": "b6", "status": "ok", "summ', stop_reason="max_tokens")
            return reply_json(WorkerResult(task_id=bid, status="ok", summary=f"done {bid}", evidence=["x"]))

        fake = FakeClaude(script, latency=0.03)
        results = await fan_out(fake.aclient, briefs, concurrency=3)
        assert isinstance(results, list) and len(results) == len(briefs), f"return one result per brief, got {results!r:.80}"
        assert all(isinstance(r, WorkerResult) for r in results), "return WorkerResult objects"
        assert [r.task_id for r in results] == [b.id for b in briefs], "keep the order of `briefs` (asyncio.gather preserves it)"
        assert fake.max_in_flight <= 3, f"peak concurrency was {fake.max_in_flight} > 3: bound the calls with asyncio.Semaphore(concurrency)"
        assert fake.max_in_flight >= 2, "the calls ran one at a time: run them concurrently (asyncio.gather) under the semaphore"
        status = {r.task_id: r.status for r in results}
        assert status["b3"] == "failed", "a 400 (anthropic.APIError) must become status='failed', not an exception"
        assert status["b6"] == "failed", "a max_tokens truncation must become status='failed' (catch ValueError / use parsed_or_raise)"
        assert all(status[b.id] == "ok" for b in briefs if b.id not in ("b3", "b6")), f"healthy workers should be ok: {status}"
        assert results[2].summary.strip(), "put the error text into summary so the synthesis can name the gap"
        assert len(fake.requests) == len(briefs), f"expected {len(briefs)} requests, saw {len(fake.requests)}"
        assert all(r.get("output_config") for r in fake.requests), "use llm.messages.parse(..., output_format=WorkerResult)"
        assert all(system_text(r) == WORKER_SYSTEM for r in fake.requests), "pass system=WORKER_SYSTEM on every call"
    await _arun_check(4, check)


await check_ex4()

### Exercise 4b — your fan-out against the real API (live, optional, ~$0.01, not scored)

Run it once Exercise 4 passes. Three tiny briefs, `concurrency=2`.

In [ ]:
# (live, optional, ~$0.01, not scored) your fan_out on the real API
if not HAS_KEY:
    print("No ANTHROPIC_API_KEY: skipping this optional live cell.")
elif not RESULTS.get(4):
    print("Finish Exercise 4 first (its checker must pass), then run this cell.")
else:
    live_briefs = [WorkerBrief(id=f"live{i}", objective=o, boundaries="one paragraph, no other topics",
                               output_format="2 bullets with evidence")
                   for i, o in enumerate(["Define prompt chaining", "Define the routing pattern",
                                          "Define the evaluator-optimizer pattern"], 1)]
    for r in await fan_out(aclient, live_briefs, concurrency=2):
        print(f"{r.task_id}: {r.status} | {r.summary[:120]}")

### Exercise 5 — Calibrate confidence buckets and route verdicts

`calibrate(labeled, target=0.95, min_samples=30)`: `labeled` is a list of `(confidence, was_real)` pairs from human-reviewed findings. Return the **set** of confidence buckets whose precision (share of `was_real=True`) is **at least** `target` **and** that have **at least** `min_samples` items. An empty list gives `set()`.

`route_verdict(verdict, auto_post)`: `verdict` is a dict with keys `"verdict"` (`confirmed` / `plausible` / `refuted`) and `"confidence"`. Return:
- `"drop_and_log"` if refuted (whatever the confidence);
- `"post_inline"` if confirmed **and** its confidence is in `auto_post`;
- `"human_triage"` otherwise.

Theory: [8.5](README.md#85-verification-passes-and-confidence-based-routing)

In [ ]:
def calibrate(labeled: list[tuple[str, bool]], target: float = 0.95, min_samples: int = 30) -> set[str]:
    """Buckets that meet the precision target on enough labeled samples."""
    # TODO
    raise NotImplementedError


def route_verdict(verdict: dict, auto_post: set[str]) -> str:
    """'post_inline' | 'human_triage' | 'drop_and_log' (rules in the task above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    def check():
        labeled = ([("high", True)] * 95 + [("high", False)] * 5          # 0.95, n=100
                   + [("medium", True)] * 40 + [("medium", False)] * 10    # 0.80, n=50
                   + [("low", True)] * 5)                                  # 1.00 but n=5
        got = calibrate(labeled)
        assert isinstance(got, set), f"return a set, got {type(got).__name__}"
        assert got == {"high"}, (f"calibrate(labeled) = {got}; expected {{'high'}}: precision >= target is inclusive, "
                                 "and a bucket with fewer than min_samples items is not evidence")
        assert calibrate(labeled, target=0.75) == {"high", "medium"}, "with target=0.75, medium (0.80, n=50) qualifies too"
        assert calibrate(labeled, min_samples=5) == {"high", "low"}, "with min_samples=5, low (1.00, n=5) qualifies"
        assert calibrate([]) == set(), "no labeled data -> nothing is auto-posted"
        cases = [
            ({"verdict": "refuted", "confidence": "high"}, {"high"}, "drop_and_log"),
            ({"verdict": "confirmed", "confidence": "high"}, {"high"}, "post_inline"),
            ({"verdict": "confirmed", "confidence": "medium"}, {"high"}, "human_triage"),
            ({"verdict": "plausible", "confidence": "high"}, {"high"}, "human_triage"),
            ({"verdict": "confirmed", "confidence": "low"}, {"high", "low"}, "post_inline"),
            ({"verdict": "confirmed", "confidence": "high"}, set(), "human_triage"),
        ]
        for v, auto, expected in cases:
            got = route_verdict(v, auto)
            assert got == expected, f"route_verdict({v}, {auto}) = {got!r}, expected {expected!r}"
    _run_check(5, check)


check_ex5()

### Exercise 6 — The adaptive planner loop's control flow

Write `adaptive_loop(goal, planner, run_subtask, max_rounds=5, max_per_round=3)`. `planner(goal, findings)` returns a `Plan` (`done`, `next_subtasks`, `open_questions`); `run_subtask(subtask)` returns a result string. Each round:

1. call `planner(goal, findings)` (that counts as one round);
2. if `plan.done` is `True` **or** `plan.next_subtasks` is empty → return `{"status": "done", "findings": findings, "rounds": <rounds so far>}` without running anything;
3. otherwise run the first `max_per_round` subtasks in order, appending `{"subtask": sub.goal, "result": run_subtask(sub)}` to `findings` for each.

After `max_rounds` planner calls without finishing, return `{"status": "budget_exhausted", "findings": findings, "rounds": max_rounds}`. Decide only on the structured `done` field: a rationale that says "done" means nothing.

Theory: [8.2](README.md#82-dynamic-adaptive-decomposition)

In [ ]:
def adaptive_loop(goal: str, planner: Callable[[str, list[dict]], Plan], run_subtask: Callable[[Subtask], str],
                  max_rounds: int = 5, max_per_round: int = 3) -> dict:
    """Planner loop: stop on plan.done / empty plan; cap subtasks per round; report budget exhaustion."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6() -> None:
    def check():
        def sub(goal, rationale="next"):
            return Subtask(goal=goal, rationale=rationale, priority="high")

        def scripted(plans):
            seen = []
            def planner(goal, findings):
                seen.append(len(findings))
                return plans[min(len(seen), len(plans)) - 1]
            return planner, seen

        ran = []
        runner = lambda s: ran.append(s.goal) or f"result of {s.goal}"

        # A: normal run with a per-round cap and a misleading rationale
        planner, seen = scripted([
            Plan(done=False, open_questions=[], next_subtasks=[sub("map", "done mapping? no, start here"), sub("rank")]),
            Plan(done=False, open_questions=[], next_subtasks=[sub(f"t{i}") for i in range(5)]),
            Plan(done=True, open_questions=[], next_subtasks=[sub("ignored")]),
        ])
        out = adaptive_loop("goal", planner, runner, max_rounds=5, max_per_round=3)
        assert isinstance(out, dict), "return a dict with status / findings / rounds"
        assert out.get("status") == "done", f"expected status 'done', got {out.get('status')!r}"
        assert ran == ["map", "rank", "t0", "t1", "t2"], f"ran {ran}: run at most max_per_round per round, in order, and nothing once done=True"
        assert seen == [0, 2, 5], f"the planner saw {seen} findings per round; pass the growing findings list every round"
        assert out.get("rounds") == 3, f"rounds should count planner calls (3), got {out.get('rounds')!r}"
        assert [f["subtask"] for f in out["findings"]] == ran and out["findings"][0]["result"] == "result of map", \
            "findings entries are {'subtask': sub.goal, 'result': run_subtask(sub)}"

        # B: never done -> budget exhausted after exactly max_rounds planner calls
        ran.clear()
        planner, seen = scripted([Plan(done=False, open_questions=[], next_subtasks=[sub("again", "all done soon")])])
        out = adaptive_loop("goal", planner, runner, max_rounds=4)
        assert out.get("status") == "budget_exhausted", f"expected 'budget_exhausted', got {out.get('status')!r}"
        assert len(seen) == 4 and out.get("rounds") == 4 and len(ran) == 4, \
            f"max_rounds=4 means 4 planner calls (saw {len(seen)}) and 4 subtasks run (saw {len(ran)})"

        # C: empty plan -> done immediately
        ran.clear()
        planner, seen = scripted([Plan(done=False, open_questions=["nothing left"], next_subtasks=[])])
        out = adaptive_loop("goal", planner, runner)
        assert out.get("status") == "done" and out.get("rounds") == 1 and ran == [], "an empty plan ends the loop (status 'done', 1 round)"
    _run_check(6, check)


check_ex6()

## Part C — Architecture scenario

**M&A due-diligence review.** A law firm's deal team must review a data room of **400 contracts** (PDFs, 5–80 pages each) within 48 hours. Partners want a risk report covering change-of-control clauses, assignment restrictions, unusual indemnities and exclusivity, plus **conflicts between documents** (for example, an amendment that contradicts its master agreement). Every finding must cite the document and clause. Partners will only read findings that have been checked; associates can triage a limited number of uncertain items. The budget is fixed, and the firm is on standard API rate limits for `claude-sonnet-5-5`.

Describe your design (15–25 lines):
1. The decomposition: which parts are a fixed pipeline, which are adaptive, and why.
2. Per-document passes vs cross-document passes: what each one receives and reports.
3. How findings are verified, how confidence is used, and how you calibrate it.
4. Concurrency, rate limits, caching, Message Batches and a rough cost estimate.
5. Failure handling and how coverage is reported to the partners.
6. Where humans sit in the loop.

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **Fixed pipeline per document** (the steps are known and identical): classify the document type → extract the clause set with structured outputs (`messages.parse` with required + nullable fields, quoted clause text and page/section for every field) → code gates (schema valid, quotes actually present in the source text, document type supported, otherwise route to an associate).
- **Adaptive only where the scope is unknown**: for example, a follow-up investigation when an extraction reveals an unexpected structure (side letters, an amendment chain), planned from findings with a round cap. Not an autonomous agent for all 400 documents.
- **Per-document local passes plus a cross-document integration pass**: the integration pass groups documents by counterparty or master agreement and receives **compact extracted clauses and parties**, not 400 PDFs, and reports only conflicts across documents (amendment vs master, inconsistent caps, conflicting exclusivity). It may request specific clauses with a tool.
- **Independent verification**: a separate instance re-checks each candidate finding against the cited clause and returns verdict + evidence + confidence. Confidence routing is **calibrated** on a labeled sample (for example, associates label the first 100 findings); only buckets that meet the precision target go to partners directly, the rest go to the associates' triage queue, and a sample of auto-accepted findings keeps being checked.
- **Throughput and cost**: bounded concurrency with a semaphore, SDK retries honoring `retry-after`, an identical cached system prefix (warm with one request first), per-call usage logging. The per-document extraction can run through **Message Batches** at 50% cost if it starts early enough in the 48 hours (one batch per pass; batches cannot chain steps). A rough estimate, for example 400 docs × ~20k input tokens and 2k output, with and without batching.
- **Failure handling**: failed or truncated documents become explicit `failed` results with a reason (scanned PDF, too long, refusal) and go to a manual list; nothing is silently dropped. The final report has a **coverage section**: documents reviewed, failed, pending human review.
- **Humans**: associates triage uncertain findings and failed documents; partners see verified findings with citations. No self-review by the extraction instance; no "drop below 7/10" gate.
</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_ex = 6
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (10 + n_ex)
print(f"Quiz:       {quiz}/10 ({quiz / 10:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS: on to Chapter 9" if overall >= 0.72 else "RETRY: reread the README sections behind your ✗ items, then try again")

Next chapter: [Chapter 9 — Escalation and Human-in-the-Loop](../09-escalation-human-in-the-loop/README.md)

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **B.** The scope is unknown and the plan must change with what is discovered: map first, prioritize, adapt (adaptive decomposition). A cannot react to the wrapper except by restarting. C confuses context size with attention. D votes away correct but inconsistently produced changes and multiplies cost.

**Q2:** **D.** A fixed chain gets its reliability from programmatic gates between steps; arithmetic belongs in code, and the gate decides on structured fields. A asks a later model step to catch an earlier step's error, probabilistically. B adds autonomy to a task with a known shape. C does not address the missing check.

**Q3:** **A.** The integration pass looks for contract, type and data-flow problems between files, so it needs interfaces and local findings, with targeted access to details. B recreates the attention dilution the per-file passes removed. C gives it nothing to check contracts against. D is arbitrary and misses most interfaces.

**Q4:** **C.** The root cause is that the reviewer carries the generator's reasoning, which already concluded the approach was fine. Only an independent instance without that context fixes it. A and B keep the same biased context. D adds useful tests but still has no independent check.

**Q5:** **B.** Only the high bucket meets the measured precision target, so only it is auto-posted; the rest stay visible to humans, and sampling catches calibration drift. A and C post buckets that miss the target. D hides real bugs (74% of medium findings were real).

**Q6:** **D.** A cache entry becomes available only after the first response starts, so a fully simultaneous fan-out cannot hit it. A caches content that never repeats. B is backwards: Haiku 4.5's minimum is 4,096 tokens, higher than Sonnet 5.5's 512, so a 3,000-token prefix would not cache at all. (The current Haiku 5.5 caches from 512 tokens, but a model switch still would not fix requests that all start at once.) C is unrelated.

**Q7:** **A.** Distinct categories with known handling are the textbook case for routing: cheap, predictable and easy to evaluate, with an explicit route for unclear input. B and C add autonomy, cost and unpredictability the task does not need. D stacks voting and an agent on top of a simple classification.

**Q8:** **C.** Union voting maximizes recall, which is what matters when misses are expensive, and the verification pass then removes false positives. A and B trade recall for precision, the wrong direction here. D throws away the other reviewers' coverage.

**Q9:** **D.** The agent already knows each action; what it lacks is a pattern for keeping parameters apart when one message holds several issues, and few-shot examples teach exactly that ([8.9](README.md#89-decomposing-multi-concern-requests)). A and C spend more on thinking or a bigger model without showing the pattern. B cannot catch the mix-up: both order IDs are in the message, so a swapped call still passes the check.

**Q10:** **A.** Anthropic's research-system write-up names shared context and heavy inter-agent dependencies as the cases where multi-agent fits poorly, which is why most coding parallelizes less well than research. B and D are false (subagents use tools and have isolated contexts), and C invents a limit; rate limits are per organization and model, not a cap of five.
</details>

<details><summary>Exercise 1 — Pick the pattern for a scenario: reference solution</summary>

```python
def choose_pattern(profile: dict) -> str:
    p = lambda key: bool(profile.get(key))
    if p("one_call_enough"):
        return "augmented_llm"
    if p("distinct_categories"):
        return "routing"
    if p("steps_known"):
        return "prompt_chaining" if p("steps_depend") else "sectioning"
    if p("iterate_with_criteria"):
        return "evaluator_optimizer"
    if p("open_ended"):
        return "agent"
    return "orchestrator_workers"


SCENARIO_PROFILES = {
    "s1": {"steps_known": True, "steps_depend": True},   # same checks, each feeds the next -> chain with gates
    "s2": {"distinct_categories": True},                 # known categories, existing processes -> routing
    "s3": {"iterate_with_criteria": True},               # clear rubric, iteration helps -> evaluator-optimizer
    "s4": {"open_ended": True},                          # unknown number of environment-driven steps -> agent
    "s5": {"steps_known": True, "steps_depend": False},  # same rubric, independent files -> sectioning
    "s6": {},                                            # subtasks decided per request, bounded -> orchestrator-workers
    "s7": {"one_call_enough": True},                     # one well-prompted call -> augmented LLM
}
```
</details>

<details><summary>Exercise 2 — Build the per-file and integration review requests: reference solution</summary>

```python
def build_local_requests(files: dict[str, str]) -> list[dict]:
    return [
        dict(model=MODEL, max_tokens=2048, system=LOCAL_RULES, output_format=FileReview,
             messages=[{"role": "user", "content": f'<file path="{path}">\n{source}\n</file>'}])
        for path, source in files.items()
    ]


def build_integration_request(local_reviews: dict[str, FileReview], pr_description: str) -> dict:
    summary = "\n".join(
        f'<file path="{path}">\n<exports>{r.exports}</exports>\n<imports>{r.imports}</imports>\n'
        f"<local_findings>{[f.issue for f in r.findings]}</local_findings>\n</file>"
        for path, r in local_reviews.items()
    )
    return dict(model=MODEL, max_tokens=2048, system=INTEGRATION_RULES, output_format=CrossFileReview,
                messages=[{"role": "user", "content": f"<pr>{pr_description}</pr>\n{summary}"}])
```
</details>

<details><summary>Exercise 3 — Build an independent reviewer request: reference solution</summary>

```python
def build_independent_review(spec: str, diff: str, criteria: str, generator_messages: list[dict]) -> dict:
    # generator_messages is deliberately unused: independence = what you leave out of `messages`.
    return dict(model=MODEL, max_tokens=2048, system=REVIEWER_SYSTEM, output_format=Review,
                messages=[{"role": "user", "content":
                           f"<spec>\n{spec}\n</spec>\n<criteria>\n{criteria}\n</criteria>\n<diff>\n{diff}\n</diff>"}])
```
</details>

<details><summary>Exercise 4 — A bounded async fan-out with partial failures: reference solution</summary>

```python
async def fan_out(llm, briefs: list[WorkerBrief], concurrency: int) -> list[WorkerResult]:
    sem = asyncio.Semaphore(concurrency)

    async def one(brief: WorkerBrief) -> WorkerResult:
        async with sem:
            try:
                response = await llm.messages.parse(
                    model=MODEL, max_tokens=1024, system=WORKER_SYSTEM, output_format=WorkerResult,
                    messages=[{"role": "user", "content": brief.model_dump_json()}])
                return parsed_or_raise(response)
            except (anthropic.APIError, ValueError, RuntimeError) as exc:
                return WorkerResult(task_id=brief.id, status="failed", summary=f"{type(exc).__name__}: {exc}", evidence=[])

    return list(await asyncio.gather(*(one(b) for b in briefs)))
```
</details>

<details><summary>Exercise 5 — Calibrate confidence buckets and route verdicts: reference solution</summary>

```python
def calibrate(labeled: list[tuple[str, bool]], target: float = 0.95, min_samples: int = 30) -> set[str]:
    buckets: dict[str, list[bool]] = {}
    for confidence, real in labeled:
        buckets.setdefault(confidence, []).append(real)
    return {c for c, v in buckets.items() if len(v) >= min_samples and sum(v) / len(v) >= target}


def route_verdict(verdict: dict, auto_post: set[str]) -> str:
    if verdict["verdict"] == "refuted":
        return "drop_and_log"
    if verdict["verdict"] == "confirmed" and verdict["confidence"] in auto_post:
        return "post_inline"
    return "human_triage"
```
</details>

<details><summary>Exercise 6 — The adaptive planner loop's control flow: reference solution</summary>

```python
def adaptive_loop(goal: str, planner: Callable[[str, list[dict]], Plan], run_subtask: Callable[[Subtask], str],
                  max_rounds: int = 5, max_per_round: int = 3) -> dict:
    findings: list[dict] = []
    for round_no in range(1, max_rounds + 1):
        plan = planner(goal, findings)
        if plan.done or not plan.next_subtasks:          # structured signal, never text
            return {"status": "done", "findings": findings, "rounds": round_no}
        for sub in plan.next_subtasks[:max_per_round]:
            findings.append({"subtask": sub.goal, "result": run_subtask(sub)})
    return {"status": "budget_exhausted", "findings": findings, "rounds": max_rounds}
```
</details>